<a href="https://colab.research.google.com/github/Zidane86-06/Data_Engineering_workshop/blob/main/Day3/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')
print(f'pandas version: {pd.__version__}')
print(f'requests version: {requests.__version__}')

All libraries imported successfully!
pandas version: 2.2.2
requests version: 2.32.4


In [ ]:
raw_df=pd.read_csv('messy_sales_data.csv')
print(f'Raw data loaded:{raw_df.shape[0]} rows,{raw_df.shape[1]} columns')
print(f'Columns:{raw_df.columns.tolist()}')
print('\nFirst 5 rows:')
raw_df.head()

Raw data loaded:30 rows,9 columns
Columns:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 5 rows:


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
print('='*55)
print('DATA QUALITY DIAGNOSIS REPORT')
print('='*55)
print('\n[1] MISSING VALES per column:')
print(raw_df.isnull().sum())
print(f'\n[2] DUPLICATE ROWS:{raw_df.duplicated().sum()}')
print('\n[3] DATA TYPES:')
print(raw_df.dtypes)
print('\n[4] UNIQUE CATEORIES:',raw_df['category'].unique())
print('[4] Sample customer names:',raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values:',raw_df['order_date'].unique()[:6])

DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS:0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [5]:
print(f"Missing values in 'quantity': {raw_df['quantity'].isnull().sum()}")

Missing values in 'quantity': 3


In [6]:
df=raw_df.copy()
print(f'Working copy created:{df.shape}')
print('raw_df is untouched -we can always reset by running df=war_df.copy()')


Working copy created:(30, 9)
raw_df is untouched -we can always reset by running df=war_df.copy()


In [7]:
print('Before fixing nulls:',df.isnull().sum().sum(),'total missing values')
df['customer_name'].fillna('Unknown customer',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f'Filled missing quantity with median:{median_qty}')
df['category'].fillna('Uncategorized',inplace=True)
print('After fixing nulls:',df.isnull().sum().sum(),'total missing values')


Before fixing nulls: 7 total missing values
Filled missing quantity with median:2.0
After fixing nulls: 1 total missing values


In [9]:
print(f'Before duplication:{len(df)} rows')
print(f'Duplicate rows:{df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']].head(6))
initial_missing_products = df['product'].isnull().sum()
if initial_missing_products > 0:
    df['product'].fillna('Unknown Product', inplace=True)
    print(f'\nFilled {initial_missing_products} missing product values with "Unknown Product".')
else:
    print('\nNo missing product values found to fill for product column.')

df.drop_duplicates(inplace=True)
print(f'\nAfter dediplication:{len(df)} rows')
print(f'Rows removed:{len(raw_df)-len(df)}')


Before duplication:30 rows
Duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Filled 1 missing product values with "Unknown Product".

After dediplication:30 rows
Rows removed:0


In [10]:
print('sample dates before parsing')
print(df['order_date'].head(8).tolist())
df['order_date']= pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)

nat_count = df['order_date'].isnull().sum()
print(f"\n Unparaseable dates(NaT): {nat_count}")

df['year'] = df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']= df['order_date'].dt.strftime('%B')

print('\n Sample dates after paring:')
print(df[['order_date','year','month','month_name']].head(5))

sample dates before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

 Unparaseable dates(NaT): 2

 Sample dates after paring:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [11]:
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')

In [12]:
df['customer_name']=(df['customer_name'].str.strip().str.title())
df['customer_name']

,customer_name
0,Ramesh Kumar
1,Priya Nair
2,Amit Verma
3,Sunita Patel
4,Ramesh Kumar
5,Kiran Mehta
6,Deepak Singh
7,Unknown Customer
8,Ananya Das
9,Vikram Iyer


In [13]:
df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')
df['order_date']

,order_date
0,2024-01-05
1,2024-01-07
2,2024-01-08
3,2024-01-10
4,2024-01-05
5,NaT
6,2024-01-12
7,2024-01-13
8,2024-01-15
9,2024-01-15


In [14]:
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['day']=df['order_date'].dt.day
df['day']

,day
0,5.0
1,7.0
2,8.0
3,10.0
4,5.0
5,NaN
6,12.0
7,13.0
8,15.0
9,15.0


In [15]:
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['day']=df['order_date'].dt.day
df['day']

,day
0,5.0
1,7.0
2,8.0
3,10.0
4,5.0
5,NaN
6,12.0
7,13.0
8,15.0
9,15.0


In [16]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce')
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['Revenue']=df['unit_price']*df['quantity']
print(df[['customer_name','product','quantity','unit_price','Revenue']].head(5))

  customer_name          product  quantity  unit_price  Revenue
0  Ramesh Kumar           Laptop       2.0       45000  90000.0
1    Priya Nair  Unknown Product       1.0       15000  15000.0
2    Amit Verma         Keyboard       3.0        1200   3600.0
3  Sunita Patel          Monitor       2.0       22000  44000.0
4  Ramesh Kumar           Laptop       2.0       45000  90000.0


In [17]:
total=df['Revenue'].sum()
print(total)

818000.0


In [19]:
print(f'original rows: {len(df)}')
# Assuming dfo was intended to be the final cleaned df
print(f'cleaned rows : {len(df)}')
print(f'rows removed : {len(raw_df)-len(df)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f"Date nulls : {df['order_date'].isnull().sum()}")
print(f"Revenue null: {df['Revenue'].isnull().sum()}")

original rows: 30
cleaned rows : 30
rows removed : 0
Missing values: 10
Duplicate rows: 0
Date nulls : 2
Revenue null: 0
